# Oppitunti 05 - Agenttinen RAG


## Asennus

Tämä muistikirja havainnollistaa Agentic RAG (Retrieval-Augmented Generation) -mallia Microsoft Agent Frameworkin avulla.

**Esivaatimukset:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — Azure AI Search -palvelun päätepisteesi
- `AZURE_SEARCH_API_KEY` — Azure AI Search API -avaimesi
- Azure OpenAI -käyttöönotto määritetty ympäristömuuttujien kautta
- Azure CLI kirjautuneena sisään (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mikä on Agentic RAG?

Perinteinen RAG noudattaa kiinteää putkea: noutaa asiakirjoja ja sitten generoi vastauksen. **Agentic RAG** menee pidemmälle antamalla agentille autonomia päättää **milloin** ja **miten** hakea tietoa.

Agentic RAG:n avulla agentti voi:
- **Päättää**, tarvitaanko haku ennen kysymykseen vastaamista
- **Valita**, mitä tietolähdettä tai työkalua kysytään
- **Arvioida** haetut tulokset ja tehdä jatkohakuja, jos ensimmäinen yritys ei ole riittävä
- **Yhdistää** tietoa useista hakuvaiheista yhdeksi yhtenäiseksi vastaukseksi

Tämä tekee agentista joustavamman ja tarkemman verrattuna staattiseen hae-sitten-generoi-putkeen.


## Hakutyökalun luominen

Agentic RAG:ssa ulkoiset tietolähteet kääritään **työkaluiksi**, joita agentti voi kutsua tarpeen mukaan. Tämä antaa agentille mahdollisuuden käsitellä tiedonhakua yhtenä toimintona muiden joukossa pakollisen vaiheen sijaan.

Alla määrittelemme matkustustietokannan ja teemme siitä työkalun, jota agentti voi kutsua etsiäkseen tietoa matkakohteista.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG-agentin rakentaminen

Nyt luomme agentin, jolle on annettu ohjeeksi **aina hakea tietoa ennen vastaamista**. Agentti käyttää `search_travel_knowledge` -työkalua perustaakseen vastauksensa tietokantaan sen sijaan, että se luottaisi omaan koulutusdataansa.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratiivinen haku — Maker-Checker-malli

Agentic RAG:n keskeinen etu on **iteratiivinen haku**. Agentti voi suorittaa useita hakukierroksia vahvistaakseen, tarkentaakseen tai laajentaakseen alkuperäisiä löydöksiään — samalla tavalla kuin "maker-checker"-työnkulku:

1. **Maker-vaihe**: Agentti hakee alkuperäiset tiedot ja laatii vastauksen.
2. **Checker-vaihe**: Agentti suorittaa lisähakuja yksityiskohtien vahvistamiseksi tai aukkojen täyttämiseksi.

Alla agentilta kysytään kysymystä, joka edellyttää useiden kohteiden vertaamista, mikä saa sen hakemaan useaan kertaan.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Yhteenveto

Tässä oppitunnissa opit rakentamaan **Agentic RAG** -järjestelmän Microsoft Agent Frameworkin avulla:

- **Agentic RAG** antaa agenteille mahdollisuuden päättää itsenäisesti, milloin tietoa haetaan, tehden hausta dynaamisen sen sijaan, että se olisi kiinteä.
- **Työkalut datalähteinä**: Ulkoiset tietovarastot (kuten Azure AI Search) kääritään työkaluiksi, joita agentti voi kutsua.
- **Iteratiivinen haku**: maker-checker -malli mahdollistaa agentin suorittaa useita hakukierroksia — etsien, varmistaen ja hienosäätäen — ennen lopullisen vastauksen tuottamista.

Tuotantoympäristössä korvaisit muistissa olevan `TRAVEL_KNOWLEDGE_BASE` -tietokannan oikealla Azure AI Search -indeksillä, joka pystyy käsittelemään laajamittaista matka-asiakirjojen hakua.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttäen tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäiskielellä tulee pitää auktoritatiivisena lähteenä. Tärkeissä asioissa suosittelemme ammattimaisen ihmiskääntäjän käyttöä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai virhetulkintatilanteista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
